In [19]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

# Libros publicados después del 1 de enero de 2000.

In [20]:
query_1 ='''SELECT COUNT(book_id) AS total_libros_recientes
FROM books
WHERE publication_date > '2000-01-01' '''


In [21]:
resultado_1 = pd.read_sql(query_1, con= engine)

In [22]:
resultado_1.head()

,total_libros_recientes
0,819


Esta métrica nos indica qué tan actualizado está el catálogo respecto al nuevo milenio.

# Número de reseñas y calificación promedio por libro.

In [23]:
query_2 = '''SELECT 
    b.title,
    COUNT(DISTINCT rev.review_id) AS total_reseñas,
    AVG(rat.rating) AS calificacion_promedio
FROM books AS b
LEFT JOIN reviews AS rev ON b.book_id = rev.book_id
LEFT JOIN ratings AS rat ON b.book_id = rat.book_id
GROUP BY b.book_id, b.title;'''

In [24]:
resultado_2= pd.read_sql(query_2, con= engine)

In [25]:
resultado_2.head()

,title,total_reseñas,calificacion_promedio
0,'Salem's Lot,2,3.666667
1,1 000 Places to See Before You Die,1,2.500000
2,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,1491: New Revelations of the Americas Before C...,2,4.500000
4,1776,4,4.000000


No siempre el libro que más gente comenta es el que mejor nota tiene. Hay libros muy famosos con muchas reseñas pero notas mediocres, y otros menos conocidos con puntajes altísimos.

# Editorial con más libros de más de 50 páginas.

In [26]:
query_3 =''' SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_libros
FROM publishers AS p
JOIN books AS b ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher_id, p.publisher
ORDER BY total_libros DESC
LIMIT 5;'''

In [27]:
resultado_3= pd.read_sql(query_3, con= engine)

In [28]:
resultado_3.head()

,publisher,total_libros
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


Este hallazgo permite concluir qué firma editorial posee la mayor capacidad de producción de obras completas, el cual es el principal proveedor de libros en la base de datos.

# Autor con la calificación promedio más alta

In [29]:
query_4 ='''SELECT 
    a.author,
    AVG(rat.rating) AS avg_rating
FROM authors AS a
JOIN books AS b ON a.author_id = b.author_id
JOIN ratings AS rat ON b.book_id = rat.book_id
WHERE b.book_id IN (
    SELECT book_id 
    FROM ratings 
    GROUP BY book_id 
    HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author_id, a.author
ORDER BY avg_rating DESC
LIMIT 5;'''

In [30]:
resultado_4 = pd.read_sql(query_4, con= engine)

In [31]:
resultado_4.head()

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.287097
1,Markus Zusak/Cao Xuân Việt Khương,4.264151
2,J.R.R. Tolkien,4.246914
3,Louisa May Alcott,4.192308
4,Rick Riordan,4.080645


In [34]:
# 1. Asegúrate de ejecutar la consulta primero
query_4 = '''
SELECT a.author, AVG(rat.rating) AS avg_rating
FROM authors AS a
JOIN books AS b ON a.author_id = b.author_id
JOIN ratings AS rat ON b.book_id = rat.book_id
WHERE b.book_id IN (
    SELECT book_id FROM ratings GROUP BY book_id HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author_id, a.author
ORDER BY avg_rating DESC
LIMIT 5;
'''

# 2. Esta es la línea que define 'df_authors'
# Asegúrate de que 'engine' esté bien configurado sin espacios en el host
df_authors = pd.read_sql(query_4, con=engine)

# 3. Ahora sí, el código de la gráfica
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
sns.barplot(x='avg_rating', y='author', data=df_authors, palette='magma')

plt.title('Top 5 Authors by Average Book Rating', fontsize=15)
plt.xlim(df_authors['avg_rating'].min() - 0.1, df_authors['avg_rating'].max() + 0.05)
plt.show()

OperationalError: (psycopg2.OperationalError) could not translate host name "yp-trainers-practicum.cluster-czs0gx-yx2d8w.us-east-1.rds.amazonaws.com" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [33]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar el estilo visual
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Crear el gráfico de barras horizontales
# Usamos 'avg_rating' para el eje X y 'author' para el eje Y
barplot = sns.barplot(
    x='avg_rating', 
    y='author', 
    data=df_authors, 
    palette='magma'
)

# Ajustar el título y etiquetas de los ejes
plt.title('Top 5 Authors by Average Book Rating\n(Minimum 50 Ratings per Book)', fontsize=15, pad=20)
plt.xlabel('Average Rating', fontsize=12)
plt.ylabel('Author', fontsize=12)

# Ajustar el límite del eje X para resaltar las diferencias
# Se enfoca en el rango de los resultados (ej. de 4.0 a 5.0)
plt.xlim(df_authors['avg_rating'].min() - 0.1, df_authors['avg_rating'].max() + 0.05)

# Añadir el valor exacto del promedio al final de cada barra
for i, v in enumerate(df_authors['avg_rating']):
    plt.text(v + 0.01, i, f'{v:.2f}', color='black', va='center', fontweight='bold')

# Ajustar el diseño para que no se corten los nombres
plt.tight_layout()

# Mostrar la gráfica
plt.show()

NameError: name 'df_authors' is not defined

<Figure size 1000x600 with 0 Axes>

El que gana aquí es el verdadero favorito del público: mucha gente lo ha leído y a casi todos les encanta. Es el autor que garantiza una buena lectura.

# Promedio de reseñas de texto de usuarios que calificaron más de 50 libros.

In [61]:
query_5 ='''WITH super_usuarios AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
)
SELECT 
    AVG(conteo_reseñas) AS promedio_reseñas_texto
FROM (
    SELECT 
        username, 
        COUNT(review_id) AS conteo_reseñas
    FROM reviews
    WHERE username IN (SELECT username FROM super_usuarios)
    GROUP BY username
) AS subconsulta;'''

In [62]:
resultado_5= pd.read_sql(query_5, con= engine)

In [63]:
resultado_5.head()

,promedio_reseñas_texto
0,24.333333


Analizamos a los usuarios que han calificado más de 50 libros (los que más leen). Vimos que, aunque puntúan muchísimos libros, no siempre se detienen a escribir una reseña larga. Esto nos ayuda a entender quiénes son los críticos que de verdad se toman el tiempo de dar su opinión detallada y quiénes solo pasan a dejar la nota rápido.